# Wrangling and EDA — Class Notes

## Part 1: Data Wrangling

**Data cleaning is approximately 80% of data science.**

### Overview: 6 Steps of Data Wrangling
1. Make the data accessible (Pandas)
2. Determine the **schema** of the data
3. Verify **data types** and **cast** as needed
4. Assess **missing values**, consider **imputation**
5. **Filter** the data: pick subsets of rows/columns
6. Save your work

### The Standard Stack
Most code starts with these imports:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

- `numpy` — extends Python with numerical analysis
- `matplotlib.pyplot` — adds plots
- `pandas` — adds data frames
- `seaborn` — simpler plotting API for quick visualizations
- `pip install package` — adds a package to your current Python environment

### (1) Dataframes
- Load the data: `df = pd.read_csv('these_data.csv', low_memory=...)`
- Verify data dimensions: `df.shape`
- Verify data integrity: `df.head()`

**Structure of a data frame:**
- **Columns are variables** — features of a phenomenon of interest
  - Create a variable from a list: `df[var] = pd.Series(list)`
  - Transform an existing variable: `df[var_transform] = transformation(df[var])`
- **Rows are observations** — instances of a phenomenon of interest
  - Create an observation from a list (must specify variables too):
    

In [ ]:
    new_obs = pd.DataFrame([[values]], columns=[var_names])
    

  - Add observations to the dataframe (**concatenate**):
    

In [ ]:
    df = pd.concat([df, new_obs], ignore_index=True)
    

- **Values** are the recorded numeric or categorical data associated with an observation for a given variable

> *"What is an observation?" is one of the most powerful questions in data science.*

### (2) Schema
- The **schema** describes how data are organized: what tables exist, what variables/columns they contain, what types those variables have, and how tables are connected
- Data are **tabular** if there are rows of observations and columns of variables in rectangular form
- In Pandas: `df.columns` returns the variables/schema

### (3) Variable Types and Casting
Six broad conceptual variable types, supported by Pandas dtypes:

- **Numeric**
  - Integer (`int`): whole numbers (…, −1, 0, 1, 2, …) → stored as `int64`
  - Real-ish (`float`): floating point numbers (e.g. 3.14159) → stored as `float64`
- **Categorical**
  - String (`str`) with a fixed list of labels (e.g. color, brand, state, customer ID, year, zip code)
  - Often stored using Pandas' `category` type
- **Logical**
  - Boolean (`bool`): `True`/`False`
- **Date**
  - `datetime`: date and time values
- **Unstructured**
  - String (`str`) without a fixed list (e.g. product reviews, doctor notes)
- **Other**
  - Object: catch-all type

**Getting to know a variable:**

In [ ]:
df[var].dtype               # current data type
df[var].isna().sum()        # number of missing values
df[var].unique()            # unique values it takes
df[var].value_counts()      # counts per unique value
df[var].hist(bins=35)       # histogram of value counts

> From this, you should be able to decide whether a given variable requires substantial cleaning.

**Type-casting / coercion:**
- Data types are often inferred "incorrectly" by the computer — e.g. a monetary value stored as `-$1,320.15` or a temperature as `23C` gets read as text because of the `$`/`C` characters.
- Forcing a variable to a different data type = **type casting** / **coercion**.
- Steps to cast a string to numeric:
  1. Replace offending symbols with blanks: `df[var] = df[var].str.replace('$', '')`
  2. Type cast: `df[var] = pd.to_numeric(df[var], errors='coerce')`
- For other casting problems: `df[var].astype(str)`, `df[var].astype(int)`, etc.

### (4) Missing Values and Imputation
- Missing data = the most important wrangling problem — values not recorded for some observations
- Mishandling/misunderstanding *why* data are missing can bias later analysis
- Missing data is **not an inconvenience** — it's information about the data-gathering process
- If needed, "fill in the blanks" with mean/median = **imputation**
- Data are **clean** when there are no missing values

**Workflow:**
1. Determine missing values per variable: `number_missing = df.isna().sum()`
2. Create missing-value dummies where appropriate: `df[f'{var}_NA'] = df[var].isna()`
3. For a numeric, consider replacing missing values with the median:
   `df[var] = df[var].fillna(df[var].median())`
   For a categorical, create a new category:
   `df[var] = df[var].fillna('NA')` or `df[var] = df[var].fillna('Missing')`
4. Finally, `df.dropna()` drops all rows with any remaining missing values

### (5) Filtering: Subsetting the Data
- **Column filtering:** given `columns = [var_1, var_2, ..., var_N]`,
  `df_cols = df[columns]` returns just those columns
- **Row filtering:** given a logical `row_condition`, e.g.
  `row_condition = (df[price] > 1000)` or `row_condition = (df[state] == 'NY')`,
  `df_rows = df[row_condition]` returns just the matching observations
- **Both at once:**
  

In [ ]:
  df_subset = df.loc[row_condition, columns]
  

> ⚠️ If you do `df = df[columns]` or `df = df[row_condition]`, you'll **lose the other columns** in `df` and have to reload the dataset from the beginning of the analysis.

### (6) Saving the Dataframe
- **Never overwrite the original datafile** — always keep the original data for reproducibility
- CSV (human-readable, widely used): `df.to_csv('data_cleaned.csv', index=False)`
- Parquet (more efficient for large/complex datasets): `df.to_parquet('data_cleaned.parquet', index=False)`

---

## Part 2: Exploratory Data Analysis (EDA)

### Outline / Big Picture
- **Random variable:** a phenomenon whose outcome is not yet known
- **Statistics / Machine Learning:** guessing future values of a phenomenon, based on past observations
- **EDA:** discovering basic features of variables
  1. Analyze categorical/numeric variables (mean/sd, median/IQR)
  2. Analyze pairs of categorical/numeric variables

### 1. Single Variables

**Recall — getting to know a variable** (same as wrangling section):

In [ ]:
df[var].dtype
df[var].isna().sum()
df[var].unique()
df[var].value_counts()
df[var].hist(bins=35)

> From this, you should be able to decide whether the variable is clean enough to proceed; otherwise, go back and clean it again.

**Statistics**
- A **statistic** is a function that maps data into a number
  - **Proportion:** what proportion of observations take a particular label?
  - **Mean:** what is the average of the recorded values of the variable?

**Analyzing a Categorical variable `C`**
- About computing how often each possible label ℓ occurs — how are the data spread across labels?
- **Sample proportion** = fraction of observations taking value ℓ
- Proportions among non-missing values: `df[var].value_counts(normalize=True)`
- Proportions including missing values as their own category: `df[var].value_counts(normalize=True, dropna=False)`
- Bar/count plot: `sns.countplot(x=df[var], stat='proportion')` or `sns.countplot(data=df, x=var, stat='proportion')`

**Analyzing a Numeric variable `x`**
- About how data are spread out across numeric values
- Visualize: `sns.histplot(df[var], bins=50)`
- Basic numeric summary: `df[var].describe()`

**Mean and Variance** — how disperse is `X = (x₁, …, xₙ)`?
- **Sample mean** (central tendency, arithmetic average):
  M(X) = (x₁ + x₂ + … + xₙ)/n = (1/n)·Σxᵢ
- **Sample variance** (average squared deviation from the mean):
  V(X) = (1/n)·Σ(xᵢ − M(X))²
- **Sample standard deviation:** sd(x) = √V(X)

**Median**
- Mean and variance are sensitive to **outliers** (atypically high/low values with a lot of leverage)
- **Robust statistics:** the study of estimators less sensitive to outliers
- **Sample median:** the value of X for which half the data are above it, half below it
  - n odd: middle value; n even: average of the two middle values
- In Pandas: `df['var'].med()`

**Quantiles**
- Median = "half below, half above." Generalize: the **f-th quantile** is the value for which proportion f of the sample is below, and (1−f) is above.
- Formula: sort x₍₁₎ ≤ x₍₂₎ ≤ … ≤ x₍ₙ₎, then q̂_f = x₍⌈nf⌉₎
- In numpy: `np.quantile(X, f)`

**Two Useful Transformations** (for long-tailed data)
- Long tails: mean is much larger/smaller than the median because a small number of very large values dominate
- **Natural logarithm:** `df['var_ln'] = np.log(df[var])`
  - Problems: goes to −∞ at zero, undefined for negative numbers
- **Inverse hyperbolic sine (arcsinh)** — better choice: `df['var_ihs'] = np.arcsinh(df[var])`
  - i(x) = ln(x + √(x² + 1))
  - Defined for positive *and* negative values, and equals zero at zero

### 2. Pairs of Variables

**Pairs of Categoricals**
- Use a **contingency table** / **cross-tabulation** to break out counts by two categoricals at once:
  `pd.crosstab(df[var1], df[var2])`
- Shows the joint counts for two categorical variables — how they vary together or not

**Pairs of Numerics**
- Visualize the relationship with a **scatterplot** (the numeric analog of a contingency table):
  `sns.scatterplot(x=df[var1], y=df[var2], alpha=.15)`
- Summarize the relationship numerically with **covariance:**
  cov(X, Y) = [Σᵢ(xᵢ − M(X))(yᵢ − M(Y))] / n
- Easily computed: `df[[var1, var2]].cov()`

**Numeric by Categorical**
- **Conditional analysis:** analyze a numeric variable `num` on a categorical variable `cat` by performing the analysis on subgroups determined by the label
- Visualize with `hue`: `sns.kdeplot(x=df[num], hue=df[cat], fill=True)`

**Group By**
- General description by groups: `df[[cat, num]].groupby(cat).describe()`
- Specific statistics: `df[[cat, num]].groupby(cat).count()` or `.mean()`, `.sum()`, `.count()`, etc.
- Custom function `fcn()`: `df[[cat, num]].groupby(cat).agg(fcn)`
- If analyzing the top/bottom of a set of observations, it often makes sense to use `.sort()` at the end